# Ejemplo de trabajo desatendido

Este notebook puede lanzarse de manera interactiva, pero está pensado para lanzarse como un trabajo (_job_) de manera desatendida.

En concreto, este trabajo calculo el número de partidos y goles por jornada. La jornada por defecto es 1, pero puede parametrizarse al lanzar el job

In [0]:
%pip install databricks-sql-connector
dbutils.library.restartPython()

In [0]:
# Parámetro de entrada del job
dbutils.widgets.text("jornada", "1", "Número de jornada")
jornada = int(dbutils.widgets.get("jornada"))

print(f"Generando informe de la jornada {jornada}")

In [0]:
import pandas as pd
from databricks import sql

with sql.connect(
    server_hostname = "<tu-workspace>.cloud.databricks.com",
    http_path       = "/sql/1.0/warehouses/<id-del-warehouse>",
    access_token    = dbutils.secrets.get(catalog="workspace", schema="liga", key="token")
) as conexion:
    with conexion.cursor() as cursor:
        cursor.execute(f"""
            SELECT equipo_local, equipo_visitante, goles_local, goles_visitante
            FROM workspace.liga.partidos
            WHERE jornada = {jornada}
        """)
        partidos = pd.DataFrame(cursor.fetchall(),
                                columns=[c[0] for c in cursor.description])

partidos["goles"] = partidos.goles_local + partidos.goles_visitante

salida = f"/Volumes/workspace/liga/datos/informe_jornada_{jornada}.csv"
partidos.to_csv(salida, index=False)

print(f"{len(partidos)} partidos, {partidos.goles.sum()} goles.")
print(f"Informe guardado en {salida}")